# Proyek Machine Learning: Clustering Analysis
**Nama:** Wahid Ivan Saputra

Dataset: Bank Transaction Dataset for Fraud Detection (Kaggle)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer

## 1. Memuat Dataset dan EDA

In [ ]:
# Memuat dataset dari Kaggle (via GitHub raw link)
url = "https://raw.githubusercontent.com/shaecodes/Analyzing-Transaction-Data-For-Fraud-Detection/main/data/bank_transactions.csv"
df = pd.read_csv(url)

# Menampilkan 5 data teratas
df.head()

In [ ]:
# Menampilkan informasi dataset
df.info()

## 2. Pembersihan dan Pra Pemprosesan Data

In [ ]:
# Menangani data yang hilang dan duplikat
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

# Feature Engineering pada Tanggal
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['PreviousTransactionDate'] = pd.to_datetime(df['PreviousTransactionDate'])

df['TransactionHour'] = df['TransactionDate'].dt.hour
df['DayOfWeek'] = df['TransactionDate'].dt.dayofweek
df['TimeSinceLast'] = (df['TransactionDate'] - df['PreviousTransactionDate']).dt.total_seconds().abs()

# Menghapus kolom ID dan kolom tanggal asli
cols_to_drop = ['TransactionID', 'AccountID', 'DeviceID', 'IP Address', 'MerchantID', 'TransactionDate', 'PreviousTransactionDate']
df = df.drop(columns=[col for col in cols_to_drop if col in df.columns])

# Encoding kategorikal
le = LabelEncoder()
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

df.head()

## 3. Membangun Model Clustering

In [ ]:
# Feature Scaling
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

# Visualisasi Elbow Method
model = KMeans(random_state=42, n_init='auto')
visualizer = KElbowVisualizer(model, k=(2,10), force_model=True)
visualizer.fit(df_scaled)
visualizer.show()

In [ ]:
# Menggunakan algoritma K-Means dengan k=6 (berdasarkan elbow method)
kmeans = KMeans(n_clusters=6, random_state=42, n_init='auto')
kmeans.fit(df_scaled)
df['Target'] = kmeans.labels_

# Menampilkan hasil cluster
df.head()

## 4. Menyimpan Hasil

In [ ]:
# Menyimpan model clustering
joblib.dump(kmeans, 'model_clustering.h5')

# Menyimpan dataset hasil clustering untuk tahap klasifikasi
df.to_csv('data_clustering.csv', index=False)

print("Model dan data berhasil disimpan!")